# 03_masking_result_view

원본 Colab 셀에서 분리. (`#@title [확인] 마스킹 결과 500장씩 + 타겟 박스 표시`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [확인] 마스킹 결과 500장씩 + 타겟 박스 표시
import os, pickle                                             # 경로 도구, 인덱스 로드 도구
import pandas as pd                                             # 매칭표 로드 도구
import matplotlib.pyplot as plt                                 # 이미지 시각화 도구
import matplotlib.patches as patches                             # 박스 그리기 도구
from PIL import Image                                             # 이미지 로드 도구

TL_DIR = "/content/drive/MyDrive/1팀 공유 문서/김태윤"              # 작업 루트
OUT = f"{TL_DIR}/masked_combo_56_18"                              # 확인할 폴더
PAGE_SIZE = 500                                                    # 한 번에 띄울 장수
PAGE = 4                                                           # 몇 번째 페이지인지 (0부터)
COLS = 25                                                          # 한 줄에 몇 장씩

with open(f"{TL_DIR}/combo_boxes_index.pkl", "rb") as f:            # (조합,각도) → 전체 알약 박스
    combo_boxes = pickle.load(f)

master = pd.read_csv(f"{TL_DIR}/tl_search_results_56_18/master_matches.csv")  # 69타겟 매칭표
TARGET_DLS = set(master["tl_dl_idx"].unique().tolist())              # 타겟 dl_idx 집합(그려야 할 것만)

files = sorted([f for f in os.listdir(OUT)                          # 폴더 내 이미지 전부
                if f.endswith((".png", ".jpg"))])
total_pages = (len(files) + PAGE_SIZE - 1) // PAGE_SIZE              # 전체 페이지 수

start = PAGE * PAGE_SIZE                                             # 이번 페이지 시작 인덱스
end = min(start + PAGE_SIZE, len(files))                             # 이번 페이지 끝 인덱스
page_files = files[start:end]                                        # 이번 페이지 파일들

print(f"전체 {len(files)}장 / 총 {total_pages}페이지 / 지금 {PAGE+1}페이지 ({start}~{end-1})")

rows = (len(page_files) + COLS - 1) // COLS                          # 필요한 행 수
fig, axes = plt.subplots(rows, COLS, figsize=(COLS*1.2, rows*1.4))    # 캔버스 준비
axes = axes.flatten()                                                  # 1차원으로

for i, fn in enumerate(page_files):                                   # 페이지 내 이미지 하나씩
    ax = axes[i]
    try:
        img = Image.open(os.path.join(OUT, fn)).convert("RGB")        # 이미지 로드
        ax.imshow(img)

        combo = fn.split("_")[0]                                      # 조합 prefix
        parts = fn.split("_")
        angle = parts[5] if len(parts) > 5 else ""                    # 각도
        boxes = combo_boxes.get((combo, angle), [])                   # 이 이미지 전체 박스

        for dl, box in boxes:                                         # 박스 하나씩
            if dl not in TARGET_DLS:                                  # 타겟 아니면 스킵(어차피 가려져있음)
                continue
            x, y, w, h = box                                          # 타겟 박스 좌표
            ax.add_patch(patches.Rectangle((x, y), w, h,               # 빨간 사각형으로 표시
                         linewidth=1, edgecolor="red", facecolor="none"))
    except Exception:
        ax.set_title("LOAD FAIL", fontsize=5)                         # 로드 실패 표시
    ax.axis("off")

for j in range(len(page_files), len(axes)):                           # 남는 칸 비우기
    axes[j].axis("off")

plt.tight_layout()
plt.show()